# Step 01_02_02 -- DuckDB Ingestion: aoestats

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoestats
**Question:** Materialise raw data into persistent DuckDB tables using the
strategies determined by 01_02_01 (union_by_name for variant columns).
**Invariants applied:** #6 (reproducibility), #7 (provenance), #9 (step scope)
**Step scope:** ingest
**Prerequisites:** 01_02_01 artifacts on disk, notebook re-executed with outputs

In [12]:
import json

from rts_predict.common.notebook_utils import get_notebook_db, get_reports_dir, setup_notebook_logging
from rts_predict.games.aoe2.config import AOESTATS_RAW_DIR
from rts_predict.games.aoe2.datasets.aoestats.ingestion import (
    load_all_raw_tables,
)

logger = setup_notebook_logging()
logger.info("Source: %s", AOESTATS_RAW_DIR)

14:27:03 INFO notebook: Source: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/raw


## 1. Ingest all DuckDB tables

Calls `load_all_raw_tables` which creates:
- `matches_raw` -- from 172 weekly match Parquet files (union_by_name)
- `players_raw` -- from 171 weekly player Parquet files (union_by_name)
- `overviews_raw` -- singleton overview.json

In [13]:
db = get_notebook_db("aoe2", "aoestats", read_only=False)
counts = load_all_raw_tables(db.con, AOESTATS_RAW_DIR)
print("Ingestion counts:")
for table, n in counts.items():
    print(f"  {table}: {n:,} rows")

14:27:03 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: Dropped existing raw_matches table.
14:27:11 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: raw_matches: 30690651 rows from /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/raw/matches/*_matches.parquet
14:27:11 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: Dropped existing raw_players table.
14:27:44 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: raw_players: 107627584 rows from /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/raw/players/*_players.parquet
14:27:44 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: Dropped existing raw_overviews table.
14:27:44 INFO rts_predict.games.aoe2.datasets.aoestats.ingestion: raw_overviews: 1 rows from /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/raw/overview/overview.json
14:27:44 

Ingestion counts:
  raw_matches: 30,690,651 rows
  raw_players: 107,627,584 rows
  raw_overviews: 1 rows


## 2. Post-ingestion validation: DESCRIBE tables

In [14]:
for table in counts:
    print(f"\n=== DESCRIBE {table} ===")
    desc_df = db.fetch_df(f'DESCRIBE "{table}"')
    print(desc_df.to_string(index=False))


=== DESCRIBE raw_matches ===
      column_name              column_type null  key default extra
              map                  VARCHAR  YES None    None  None
started_timestamp TIMESTAMP WITH TIME ZONE  YES None    None  None
         duration                   BIGINT  YES None    None  None
     irl_duration                   BIGINT  YES None    None  None
          game_id                  VARCHAR  YES None    None  None
          avg_elo                   DOUBLE  YES None    None  None
      num_players                   BIGINT  YES None    None  None
       team_0_elo                   DOUBLE  YES None    None  None
       team_1_elo                   DOUBLE  YES None    None  None
  replay_enhanced                  BOOLEAN  YES None    None  None
      leaderboard                  VARCHAR  YES None    None  None
           mirror                  BOOLEAN  YES None    None  None
            patch                   BIGINT  YES None    None  None
   raw_match_type               

## 3. NULL rates on key fields

In [15]:
# matches_raw NULL rates
match_null_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(game_id) AS game_id_null,
    COUNT(*) - COUNT(map) AS map_null,
    COUNT(*) - COUNT(started_timestamp) AS started_timestamp_null,
    COUNT(*) - COUNT(filename) AS filename_null
FROM matches_raw
"""
print("=== matches_raw NULL rates ===")
print(db.fetch_df(match_null_query).to_string(index=False))

=== raw_matches NULL rates ===
 total_rows  game_id_null  map_null  started_timestamp_null  filename_null
   30690651             0         0                       0              0


In [16]:
# players_raw NULL rates
player_null_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(game_id) AS game_id_null,
    COUNT(*) - COUNT(profile_id) AS profile_id_null,
    COUNT(*) - COUNT(winner) AS winner_null,
    COUNT(*) - COUNT(filename) AS filename_null
FROM players_raw
"""
print("=== players_raw NULL rates ===")
print(db.fetch_df(player_null_query).to_string(index=False))

=== raw_players NULL rates ===
 total_rows  game_id_null  profile_id_null  winner_null  filename_null
  107627584             0             1185            0              0


In [17]:
# overviews_raw row count
print(f"overviews_raw: {counts.get('overviews_raw', 'N/A'):,} rows")

raw_overviews: 1 rows


In [18]:
db.close()

## 4. Write artifacts

In [19]:
reports_dir = get_reports_dir("aoe2", "aoestats")
artifacts_dir = reports_dir / "artifacts" / "01_exploration" / "02_eda"
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_data = {
    "step": "01_02_02",
    "dataset": "aoestats",
    "tables_created": {
        table: {"row_count": n} for table, n in counts.items()
    },
}

artifact_path = artifacts_dir / "01_02_02_duckdb_ingestion.json"
artifact_path.write_text(json.dumps(artifact_data, indent=2))
logger.info("Artifact written: %s", artifact_path)

14:27:45 INFO notebook: Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/02_eda/01_02_02_duckdb_ingestion.json


In [20]:
md_lines = [
    "# Step 01_02_02 -- DuckDB Ingestion: aoestats\n",
    "",
    "## Tables created\n",
    "",
    "| Table | Rows |",
    "|-------|------|",
]
for table, n in counts.items():
    md_lines.append(f"| {table} | {n:,} |")
md_lines.extend([
    "",
    "## Ingestion strategy\n",
    "",
    "- `matches_raw` and `players_raw`: `union_by_name = true` to handle",
    "  variant columns across weekly files.",
    "- `overviews_raw`: `read_json_auto` on singleton overview.json.",
    "- All tables include `filename` provenance column.",
    "",
    "## SQL used\n",
    "",
    "See `src/rts_predict/games/aoe2/datasets/aoestats/ingestion.py` for all",
    "SQL constants.",
])

md_path = artifacts_dir / "01_02_02_duckdb_ingestion.md"
md_path.write_text("\n".join(md_lines))
logger.info("Report written: %s", md_path)

14:27:45 INFO notebook: Report written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/02_eda/01_02_02_duckdb_ingestion.md
